In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error ,mean_absolute_error ,r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,SimpleRNN,Dropout,Dense

In [ ]:
data = pd.read_csv('Google_Stock_Price_Train.csv',thousands=",")
data.head()

In [ ]:
data.info()

In [ ]:
data['Date']= pd.to_datetime(data['Date'])

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.plot(x='Date',y=['Open','High','Low','Close'],figsize=(12,6))
plt.grid(True)
plt.show()

In [ ]:
data.drop('Date',axis=1,inplace=True)

In [ ]:
data.info()

In [ ]:
# normalize the data using MinMaxScaler
sc =MinMaxScaler()
data=sc.fit_transform(data)

In [ ]:
# 60 days of historical data to predict the next day's price
def create_seq (data,idx,seqlen=60):
    # seqlen is the number of previous time steps to consider for predicting the next value
    x,y=[],[]
    for i in range(seqlen,len(data)):
        x.append(data[i-seqlen:i])
        y.append(data[i,idx])
    return np.array(x),np.array(y)

x,y= create_seq(data,3)

x

In [ ]:
y

In [ ]:
split = int(0.8*len(data)) # 80% for training and 20% for testing
x_train = x[:split]
x_test = x[split:]
y_train = y[:split]
y_test = y[split:]

In [ ]:
# using SimpleRNN
# SimpleRNN is a type of recurrent neural network that is used for processing sequential data. 
# It has a simple architecture and is suitable for tasks where the relationships between time steps are not too complex. 
# In this model, we will use multiple layers of SimpleRNN to capture the temporal dependencies in the stock price data, 
# along with Dropout layers to prevent overfitting. 
# Finally, we will add a Dense layer to output the predicted stock price.

model_sr = Sequential()

# The first layer is a SimpleRNN layer with 64 units, which takes the input shape of (x_train.shape[1], x_train.shape[2]). 
# The return_sequences=True argument allows the layer to return the full sequence of outputs, which is necessary for stacking multiple RNN layers. 
# The activation function used is 'tanh', which is commonly used in RNNs to introduce non-linearity. 
# After the SimpleRNN layer, we add a Dropout layer with a rate of 0.2 to prevent overfitting by randomly setting 20% of the input units to 0 during training.
model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh',input_shape=(x_train.shape[1],x_train.shape[2])))
model_sr.add(Dropout(0.2))

model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh'))
model_sr.add(Dropout(0.2))

model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh'))
model_sr.add(Dropout(0.2))

model_sr.add(SimpleRNN(64))
model_sr.add(Dropout(0.2))

# Finally, we add a Dense layer with 1 unit to output the predicted stock price.
model_sr.add(Dense(1))

# Compile the model using the Adam optimizer and mean squared error loss function, 
# which is suitable for regression tasks like stock price prediction.
model_sr.compile(optimizer='adam',loss='mse')

# Print the summary of the model architecture, which includes the number of layers, 
# the number of parameters in each layer, and the total number of parameters in the model.
model_sr.summary()

In [ ]:
# h_sr stands for history of the SimpleRNN model training process. 
# It contains information about the training and validation loss and metrics for each epoch.
h_sr= model_sr.fit(
    x_train,y_train,
    batch_size=32,
    epochs=50,
    validation_data=(x_test,y_test),
    verbose=1
)

In [ ]:
y_pred= model_sr.predict(x_test)

In [ ]:
close_sc = MinMaxScaler()
close_sc.min_,close_sc.scale_ = sc.min_[3:4],sc.scale_[3:4]

In [ ]:
# Inverse transform the predicted and actual values to get the original scale of stock prices
predicted = close_sc.inverse_transform(y_pred)
actual = close_sc.inverse_transform(y_test.reshape(-1,1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual,label="Actual")
plt.plot(predicted,label="Predicted")
plt.grid(True)
plt.show()

In [ ]:
# Actual prediction error
print("RMSE : ",np.sqrt(mean_squared_error(actual,predicted)))
# How much variance is explained by the model
print("R2 score ",r2_score(actual,predicted))
#
print("MAE : ",mean_absolute_error(actual,predicted))

In [ ]:
# using LSTM
# LSTM (Long Short-Term Memory) is a type of recurrent neural network that is designed to capture long-term dependencies in sequential data. 
# It has a more complex architecture than SimpleRNN, with memory cells and gating mechanisms that allow it to retain information over longer time periods. 
# In this model, we will use multiple layers of LSTM to capture the temporal dependencies in the stock price data, along with Dropout layers to prevent overfitting. 
# Finally, we will add a Dense layer to output the predicted stock price.
model_lstm = Sequential()

# The first layer is an LSTM layer with 64 units, 
# which takes the input shape of (x_train.shape[1], x_train.shape[2]).
model_lstm.add(LSTM(64,return_sequences=True,input_shape=(x_train.shape[1],x_train.shape[2])))
model_lstm.add(Dropout(0.2))

model_lstm.add(LSTM(64))
model_lstm.add(Dropout(0.1))

# Finally, we add a Dense layer with 1 unit to output the predicted stock price.
model_lstm.add(Dense(1))

model_lstm.compile(optimizer='adam',loss='mse')

model_lstm.summary()

In [ ]:
# h_lstm stands for history of the LSTM model training process.
h_lstm= model_lstm.fit(
    x_train,y_train,
    batch_size=32,
    epochs=50,
    validation_data=(x_test,y_test),
    verbose=1
)

In [ ]:
y_pred_lstm= model_lstm.predict(x_test)

In [ ]:
# Inverse transform the predicted and actual values to get the original scale of stock prices
predicted_lstm = close_sc.inverse_transform(y_pred_lstm)
actual_lstm = close_sc.inverse_transform(y_test.reshape(-1,1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_lstm,label="Actual")
plt.plot(predicted_lstm,label="Predicted")
plt.grid(True)
plt.show()

In [ ]:
# Evaluate the performance of the LSTM model using RMSE, R2 score, and MAE
print("RMSE : ",np.sqrt(mean_squared_error(actual_lstm,predicted_lstm)))
print("R2 score ",r2_score(actual_lstm,predicted_lstm))
print("MAE : ",mean_absolute_error(actual_lstm,predicted_lstm))

# 1. Difference between CNN and RNN

| CNN | RNN |
|---|---|
| Used for image data | Used for sequential/time-series data |
| Extracts spatial features | Learns temporal dependencies |
| Best for image classification | Best for stock prediction and NLP |

## CNN Applications
- Image classification
- Face recognition
- Object detection

## RNN Applications
- Stock prediction
- NLP
- Speech recognition

---

# 2. Input Dimensions of RNN

RNN input must be a 3D tensor.

## Input Shape
```text
(batch_size, time_steps, features)
```

## Meaning
- Batch Size → Number of samples
- Time Steps → Previous observations
- Features → Input variables

## Example
```python
(32, 60, 1)
```

Means:
- 32 samples
- 60 timestamps
- 1 feature

## Output Shape

### return_sequences=False
```text
(batch_size, units)
```

### return_sequences=True
```text
(batch_size, time_steps, units)
```

---

# 3. Main Difficulties in Training RNNs

- Vanishing gradient problem
- Exploding gradient problem
- Difficulty learning long-term dependencies
- Slow training
- High computational cost

---

# 4. Uses of RNN in NLP

RNN is used for sequential language data.

## Applications
- Text generation
- Chatbots
- Sentiment analysis
- Language translation
- Speech recognition
- Next word prediction

---

# 5. Difference between Feedforward Network and RNN

| Feedforward Network | RNN |
|---|---|
| No memory | Has memory |
| Processes independent data | Processes sequential data |
| No feedback connections | Has recurrent connections |

---

# 6. Output Calculation in RNN

RNN output depends on:
- Current input
- Previous hidden state

## Formula
```text
h_t = f(Wx_t + Uh_{t-1} + b)
```

Where:
- \( h_t \) → Current hidden state
- \( x_t \) → Current input
- \( h_{t-1} \) → Previous hidden state

---

# 7. Difference between RNN and LSTM

| RNN | LSTM |
|---|---|
| Short-term memory | Long-term memory |
| Suffers from vanishing gradient | Handles long dependencies |
| Simpler architecture | More advanced architecture |

## Advantages of LSTM
- Better for stock prediction
- Learns long-term dependencies
- More stable training

---

# Basic Viva Questions

## What is RNN?
RNN is a neural network designed for sequential and time-series data.

---

## What is time-series data?
Data collected over time intervals where sequence matters.

Example:
- Stock prices
- Weather data

---

## What is stock price prediction?
Predicting future stock prices using historical data.

---

## Why is RNN suitable for time-series data?
Because RNN remembers previous information using hidden states.

---

## What is sequential data?
Data where previous observations affect future values.

---

## What is timestep in RNN?
Number of previous observations used for prediction.

---

## What is hidden state?
Hidden state stores previous sequence information.

---

## What activation function is used in RNN?
Tanh activation function.

---

## Why is normalization important?
- Faster training
- Better convergence
- Stable learning

---

## What is sliding window technique?
Using previous timestamps to predict future values.

Example:
```text
Previous 60 days → Predict next day
```

---

# Intermediate Viva Questions

## What is return_sequences in RNN?

### return_sequences=True
Returns output for all timesteps.

### return_sequences=False
Returns only final output.

---

## Why use Dropout in RNN?
To reduce overfitting.

---

## What is overfitting?
Model performs well on training data but poorly on unseen data.

---

## Why is tanh used in RNN?
Because it outputs values between:
```text
-1 to 1
```

---

## What is sequence learning?
Learning patterns from sequential data.

---

## Difference between SimpleRNN and LSTM

| SimpleRNN | LSTM |
|---|---|
| Short memory | Long memory |
| Simpler | More advanced |

---

## Difference between LSTM and GRU

| LSTM | GRU |
|---|---|
| More complex | Simpler |
| More gates | Fewer gates |

---

## What optimizer is commonly used?
Adam optimizer.

---

## Why is Adam optimizer popular?
- Fast convergence
- Adaptive learning rate
- Efficient training

---

## What loss function is used in stock prediction?
Mean Squared Error (MSE).

---

## Explain RMSE
Measures average prediction error.

Lower RMSE = Better model.

---

## Explain MAE
Measures average absolute error.

---

## Explain R² Score
Measures how well the model explains variance.

Closer to 1 = Better model.

---

## What is train-test split?
Dividing dataset into:
- Training data
- Testing data

---

## Why is scaling important?
Prevents large values from dominating training.

---

# Advanced Viva Questions

## Explain forward propagation in RNN
Input sequence passes through hidden layers to generate output.

---

## Explain Backpropagation Through Time (BPTT)
Updates RNN weights using gradients across timesteps.

---

## What are trainable parameters?
Weights and biases learned during training.

---

## Why does RNN suffer from vanishing gradients?
Gradients become very small during repeated backpropagation.

---

## What is memory cell in LSTM?
Stores long-term information.

---

## What are gates in LSTM?
- Forget gate
- Input gate
- Output gate

---

## Why is LSTM preferred for stock prediction?
Because it handles long-term dependencies better.

---

## What is recurrent connection?
Previous output is fed back into the network.

---

## Why is RNN slower than CNN?
Because RNN processes data sequentially.

---

## What evaluation metrics are used?
- RMSE
- MAE
- R² Score

---

# Common Practical Questions

## Why did you choose RNN for stock prediction?
Because stock data is sequential time-series data.

---

## Why normalize stock dataset?
To improve model training and stability.

---

## Why use multiple RNN layers?
To learn complex sequential patterns.

---

## Why use Dropout layers?
To reduce overfitting.

---

## Why use Dense layer at the end?
To generate final stock price prediction.

---

## What challenges occur in stock prediction?
- Market volatility
- Noise in data
- Sudden market changes